In [1]:
import boto3
import json
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime
import os
from dotenv import load_dotenv
import re
import uuid
import mlflow
import dagshub

# Load env file
load_dotenv()

os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("DAGSHUB_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("DAGSHUB_TOKEN")

# Tracking server
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment("inframind_experiment")

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name="ap-south-1",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY
)

x:\nasim_xhqpjmy\Code\GenAI\InfraMind\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Llama 3 Pricing on AWS Bedrock (On-Demand per 1k tokens)
PRICING = {
    "meta.llama3-8b-instruct-v1:0": {"input": 0.0003, "output": 0.0006},
    "meta.llama3-70b-instruct-v1:0": {"input": 0.00265, "output": 0.0035}
}

In [3]:
class RCAOutput(BaseModel):
    incident_id: str = Field(description="Unique ID for the incident")
    severity: str = Field(description="Critical, High, Medium, or Low")
    summary: str = Field(description="One-sentence summary of what happened")
    root_cause: str = Field(description="The primary technical reason for the failure")
    immediate_fix: str = Field(description="What to do right now to restore service")
    confidence_score: float = Field(description="AI's confidence (0.0 to 1.0)")
    model_used: str = Field(description="Llama-3-8B or Llama-3-70B")

json_schema = RCAOutput.model_json_schema()

In [4]:
def track_usage(response_body, model_id):
    """Calculates cost and logs tokens to MLflow"""
    i_tokens = response_body.get("prompt_token_count", 0)
    o_tokens = response_body.get("generation_token_count", 0)
    
    cost = (i_tokens * PRICING[model_id]["input"]) + (o_tokens * PRICING[model_id]["output"])
    
    # Log to MLflow (Assuming a run is active)
    mlflow.log_metric(f"tokens_in", i_tokens)
    mlflow.log_metric(f"tokens_out", o_tokens)
    mlflow.log_metric(f"cost_usd", cost)
    
    return cost

In [5]:
def extract_json(text: str) -> str:
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON object found in model output")
    return text[start:end + 1]

In [6]:
def call_llama(prompt, model, max_tokens=512):
    response = bedrock_runtime.invoke_model(
        modelId=model,
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "prompt": prompt,
            "max_gen_len": max_tokens,
            "temperature": 0.1,
            "top_p": 0.9
            # "stop": ["\n\n"]
        })
    )

    result = json.loads(response["body"].read())
    track_usage(result, model)

    return result["generation"]

In [7]:
def investigate_incident(log, context, model):

    prompt = f"""
    <|begin_of_text|>

    <|start_header_id|>system<|end_header_id|>
    You are a Site Reliability Engineer investigating an infrastructure incident.

    Analyze the log carefully.

    IMPORTANT RULES:
    - Only use facts visible in the log.
    - Do NOT invent causes.
    - Use the runbook context to identify known failure patterns.

    Return structured bullet points under these sections:

    Symptoms:
    - observable errors or behavior from the log

    Failing Component:
    - service or infrastructure element likely failing

    Infrastructure Layer:
    - Application / Database / Network / Kubernetes / Cloud / Unknown

    Possible Causes (based on log evidence and runbook):
    - list plausible technical causes

    <|eot_id|>

    <|start_header_id|>user<|end_header_id|>
    Runbook context:
    {context}

    Incident log:
    {log}

    <|eot_id|>

    <|start_header_id|>assistant<|end_header_id|>
    """

    return call_llama(prompt, model)

In [8]:
def infer_root_cause(investigation, context, model):

    prompt = f"""
<|begin_of_text|>

<|start_header_id|>system<|end_header_id|>
You are a senior Site Reliability Engineer performing root cause analysis.

Use the RUNBOOK CONTEXT as the primary source of truth.

Rules:
- Root cause MUST be supported by the runbook.
- If the runbook suggests known failure modes, prioritize them.
- Do NOT invent causes not mentioned in the runbook.
- Use the investigation summary as evidence.

Explain reasoning step by step.

Output sections:

Root Cause Analysis:
- explanation

Evidence:
- log symptoms supporting the diagnosis

Runbook Reference:
- which runbook guidance supports this diagnosis

<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Investigation summary:
{investigation}

Runbook context:
{context}

<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
"""

    return call_llama(prompt, model)

In [9]:
def generate_fix(reasoning, model):

    prompt = f"""
    <|begin_of_text|>

    <|start_header_id|>system<|end_header_id|>
    You are a Site Reliability Engineer writing remediation instructions.

    Provide concrete operational steps.

    Return three sections:

    Immediate Mitigation:
    - actions to restore service now

    Long-term Fix:
    - configuration or architecture fixes

    Verification Steps:
    - commands or checks to confirm the issue is resolved

    Do not leave any section empty.
    Do not use placeholders.

    <|eot_id|>

    <|start_header_id|>user<|end_header_id|>
    Root cause reasoning:
    {reasoning}
    <|eot_id|>

    <|start_header_id|>assistant<|end_header_id|>
    """
    return call_llama(prompt, model)

In [10]:
def format_rca(reasoning, fix, incident_id, model):

    prompt = f"""
<|begin_of_text|>

<|start_header_id|>system<|end_header_id|>
Convert the RCA analysis into JSON.

Rules:
- Fill all fields with meaningful information.
- Use the remediation instructions for "immediate_fix".

Schema:
{json.dumps(json_schema)}

<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Incident ID:
{incident_id}

Root cause reasoning:
{reasoning}

Remediation instructions:
{fix}

<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
"""

    result = call_llama(prompt, model)

    json_text = extract_json(result.strip())

    data = json.loads(json_text)

    rca = RCAOutput.model_validate(data)

    rca.incident_id = incident_id

    return rca

In [11]:
def generate_infra_rca(raw_log: str, context: str = "", feedback: str = "") -> RCAOutput:

    incident_id = str(uuid.uuid4())

    selected_model = (
        "meta.llama3-70b-instruct-v1:0"
        if len(raw_log) > 2000
        else "meta.llama3-8b-instruct-v1:0"
    )

    print("🔎 Investigator Agent...")
    investigation = investigate_incident(raw_log, context, selected_model)

    if feedback:
        investigation = investigation + f"\n\nSenior SRE critique:\n{feedback}"

    print("🧠 Root Cause Agent...")
    reasoning = infer_root_cause(investigation, context, selected_model)

    print("🔧 Fix Agent...")
    fix = generate_fix(reasoning, selected_model)

    print("📋 Formatter Agent...")
    rca = format_rca(reasoning, fix, incident_id, selected_model)

    model_label = "Llama-3-70B" if "70b" in selected_model else "Llama-3-8B"
    rca.model_used = model_label

    return rca

In [12]:
test_log = "ERROR: [Kubelet] Failed to pull image 'nginx:latest': RPC error: code = Unknown desc = Error response from daemon: Get https://registry-1.docker.io/v2/: net/http: TLS handshake timeout"

try:
    analysis = generate_infra_rca(test_log, context="VPC Security Groups allow outbound 443.")
    print("\n--- RCA REPORT ---")
    print("Incident ID:", analysis.incident_id)
    print("Severity:", analysis.severity)
    print("Summary:", analysis.summary)
    print("Root Cause:", analysis.root_cause)
    print("Immediate Fix:", analysis.immediate_fix)
    print("Confidence:", analysis.confidence_score)
    print("Model:", analysis.model_used)
except Exception as e:
    print(f"Mission Failed: {e}")

🔎 Investigator Agent...
🧠 Root Cause Agent...
🔧 Fix Agent...
📋 Formatter Agent...

--- RCA REPORT ---
Incident ID: 13616dcf-ae0a-4170-9ef7-2fec97145d0b
Severity: High
Summary: VPC Security Groups not allowing inbound traffic on port 443
Root Cause: VPC Security Groups not allowing inbound traffic on port 443
Immediate Fix: Check the VPC Security Group configuration for the specific security group associated with the instance, verify that the security group allows inbound traffic on port 443, and update the security group to allow inbound traffic on port 443 if necessary.
Confidence: 0.9
Model: Llama-3-8B


In [13]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockEmbeddings

# Load documents
loader = DirectoryLoader("runbook", glob="*.md", loader_cls=TextLoader)
docs = loader.load()

print("Docs:", len(docs))

headers_to_split_on = [
    ("#", "Header1"),
    ("##", "Header2"),
    ("###", "Header3"),
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

splits = []

for doc in docs:
    splits.extend(splitter.split_text(doc.page_content))

print("Splits:", len(splits))

embeddings = BedrockEmbeddings(
    client=bedrock_runtime,
    model_id="amazon.titan-embed-text-v2:0"
)

vector_db = FAISS.from_documents(
    splits,
    embeddings
)

def get_context(query):
    results = vector_db.similarity_search(query, k=4)
    return "\n---\n".join([doc.page_content for doc in results])

Docs: 11
Splits: 106


In [14]:
from langchain_aws import ChatBedrock
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

class BedrockJudge(DeepEvalBaseLLM):
    def __init__(self):
        self.model_id = "mistral.mistral-7b-instruct-v0:2"

    def load_model(self):
        return self

    def generate(self, prompt: str) -> str:
        # Wrap in Mistral format
        mistral_prompt = f"<s>[INST] {prompt} [/INST]"
        
        response = bedrock_runtime.invoke_model(
            modelId=self.model_id,
            contentType="application/json",
            accept="application/json",
            body=json.dumps({
                "prompt": mistral_prompt,
                "max_tokens": 1024,
                "temperature": 0.0
            })
        )
        result = json.loads(response["body"].read())
        return result.get("outputs", [{}])[0].get("text", "")

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self) -> str:
        return "Bedrock Mistral-7B Judge"

# Instantiate once — reused across all evaluations
bedrock_judge = BedrockJudge()

In [15]:
def run_deepeval(log, context, output_json):
    try:
        test_case = LLMTestCase(
            input=log,
            actual_output=output_json.immediate_fix,
            retrieval_context=[context]
        )

        # Pass bedrock_judge so DeepEval never tries to call OpenAI
        faith_metric = FaithfulnessMetric(threshold=0.7, model=bedrock_judge)
        rel_metric = AnswerRelevancyMetric(threshold=0.7, model=bedrock_judge)

        faith_metric.measure(test_case)
        rel_metric.measure(test_case)

        return faith_metric.score, rel_metric.score

    except Exception as e:
        print(f"DeepEval evaluation failed: {e}")
        return 0.0, 0.0

In [16]:
def senior_sre_critique(rca_report: RCAOutput, runbook_context: str):
    
    # Mistral instruct format
    critique_prompt = f"""<s>[INST]
    You are a Staff SRE reviewing an RCA report for technical accuracy.
    Score the RCA from 0-10.
    Return output strictly as:

    SCORE: [X] | NOTE: explanation

    Runbook context:
    {runbook_context}

    RCA Report:
    {rca_report.model_dump_json()}
    [/INST]"""

    response = bedrock_runtime.invoke_model(
        modelId="mistral.mistral-7b-instruct-v0:2",
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "prompt": critique_prompt,
            "max_tokens": 512,        
            "temperature": 0.1
        })
    )
    
    result = json.loads(response['body'].read())
    
    print("\n--- RAW CRITIQUE RESPONSE ---")
    print(result)
    
    # Mistral response format is also different
    critique_text = result.get("outputs", [{}])[0].get("text", "")
    return critique_text

In [17]:
def run_autonomous_workflow(log_text, max_retries=2):
    if mlflow.active_run():
        mlflow.end_run()

    with mlflow.start_run(run_name=f"Incident_{datetime.now().strftime('%H%M%S')}"):
        mlflow.log_param("incident_log", log_text)
        
        knowledge = get_context(f"incident: {log_text} troubleshooting runbook")
        mlflow.log_text(knowledge, "retrieved_context.txt")
        mlflow.log_param("retrieved_context_length", len(knowledge))
        current_feedback = ""
        
        for attempt in range(max_retries + 1):
            print(f"\n🚀 Execution Attempt {attempt + 1}...")
            
            # Generate RCA
            rca = generate_infra_rca(log_text, context=knowledge, feedback=current_feedback)
        
            mlflow.log_metric(f"confidence_score_attempt_{attempt+1}", rca.confidence_score)
            mlflow.log_metric("rca_length", len(rca.summary))
            
            mlflow.log_dict({
                "incident_id": rca.incident_id,
                "severity": rca.severity,
                "summary": rca.summary,
                "root_cause": rca.root_cause,
                "immediate_fix": rca.immediate_fix
            }, f"rca_output_attempt_{attempt+1}.json")
            
            # Senior SRE critique
            critique = senior_sre_critique(rca, knowledge)
            print(f"Critique Received: {critique.strip()}")
            mlflow.log_text(critique, f"critique_attempt_{attempt+1}.txt")

            faith_score, rel_score = run_deepeval(
                        log_text,
                        knowledge,
                        rca)
            
            mlflow.log_metric(f"faithfulness_score_attempt_{attempt+1}", faith_score)
            mlflow.log_metric(f"relevancy_score_attempt_{attempt+1}", rel_score)

            # Parse Score
            match = re.search(r"score\s*:\s*\[?(\d+)\]?", critique, re.IGNORECASE)
            score = float(match.group(1)) / 10 if match else 0.0
            
            mlflow.log_metric(f"attempt_{attempt+1}_score", score)
            
            
            # 4. Check quality threshold
            
            if score >= 0.8:
                print(f"✅ Success! Quality score {score} meets threshold.")
                mlflow.log_dict(rca.model_dump(), "final_rca_output.json")
                mlflow.log_text(critique, "final_senior_sre_review.txt")
                return rca
            else:
                print(f"⚠️ Score too low ({score}). Triggering self-correction loop...")
                current_feedback = critique # Pass the bad grade back into the AI
                
        print("🚨 Max retries reached. Returning best effort.")
        return rca

In [18]:
if __name__ == "__main__":
    new_incident_log = "ERROR 500: Database connection refused while connecting to host rds.cluster-inframind.internal"
    
    try:
        final_rca = run_autonomous_workflow(new_incident_log)
        
        print("\n--- FINAL ANALYTICS REPORT ---")
        print("Incident ID:", final_rca.incident_id)
        print("Severity:", final_rca.severity)
        print("Summary:", final_rca.summary)
        print("Root Cause:", final_rca.root_cause)
        print("Immediate Fix:", final_rca.immediate_fix)
        print("Confidence:", final_rca.confidence_score)
        print("Model:", final_rca.model_used)
        
    except Exception as e:
        print(f"System Failure: {e}")

🏃 View run agreeable-crow-254 at: https://dagshub.com/nasim-raj-laskar/InfraMind.mlflow/#/experiments/0/runs/7603f09a7acf4c71ac7d2f33c8edee43
🧪 View experiment at: https://dagshub.com/nasim-raj-laskar/InfraMind.mlflow/#/experiments/0

🚀 Execution Attempt 1...
🔎 Investigator Agent...
🧠 Root Cause Agent...
🔧 Fix Agent...
📋 Formatter Agent...

--- RAW CRITIQUE RESPONSE ---
{'outputs': [{'text': ' SCORE: 10 | NOTE: The RCA report accurately identifies the root cause and provides appropriate immediate fixes for a connection refused error caused by an RDS instance being stopped, rebooting, or in maintenance window. The steps in the diagnosis section are clear and well-documented. The only suggestion for improvement would be to include more specific error messages or logs to help differentiate between different types of database connection errors.', 'stop_reason': 'stop'}]}
Critique Received: SCORE: 10 | NOTE: The RCA report accurately identifies the root cause and provides appropriate immedi

x:\nasim_xhqpjmy\Code\GenAI\InfraMind\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

✅ Success! Quality score 1.0 meets threshold.
🏃 View run Incident_171710 at: https://dagshub.com/nasim-raj-laskar/InfraMind.mlflow/#/experiments/0/runs/fa60c70c893543d78a2600981a366613
🧪 View experiment at: https://dagshub.com/nasim-raj-laskar/InfraMind.mlflow/#/experiments/0

--- FINAL ANALYTICS REPORT ---
Incident ID: 5d56e781-3957-4651-8d2e-b0d22ab59e2f
Severity: Medium
Summary: RDS instance stopped, rebooting, or in maintenance window causing connection refused error
Root Cause: RDS instance stopped, rebooting, or in maintenance window
Immediate Fix: Check RDS instance status using `aws rds describe-db-instances`, start instance if stopped, wait for reboot to complete if rebooting, wait for maintenance window to complete if in maintenance window
Confidence: 0.9
Model: Llama-3-8B
